# Lesson 25: Reinforcement Learning with Gymnasium (`CartPole-v1`)

Use open-source **`gymnasium`**, discretize continuous 4D CartPole observations, train a sparse dictionary Q-Table with `Qbest` checkpointing, and plot the 100-episode running average.

In [ ]:
!pip install -q gymnasium numpy matplotlib

In [ ]:
import random
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

env = gym.make('CartPole-v1')
def discretize(x):
    return tuple((x / np.array([0.25, 0.25, 0.01, 0.1])).astype(int))

Q = {}
actions = (0, 1)
def qvalues(s): return [Q.get((s, a), 0.0) for a in actions]

alpha, gamma, epsilon = 0.3, 0.9, 0.90
rewards = []
for ep in range(3000):
    obs, _ = env.reset()
    done = False
    total = 0
    while not done:
        s = discretize(obs)
        if random.random() < epsilon:
            a = int(np.argmax(qvalues(s)))
        else:
            a = env.action_space.sample()
        nobs, rew, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        total += rew
        ns = discretize(nobs)
        Q[(s, a)] = (1 - alpha) * Q.get((s, a), 0.0) + alpha * (rew + gamma * max(qvalues(ns)))
        obs = nobs
    rewards.append(total)
env.close()

running_avg = np.convolve(rewards, np.ones(100)/100, mode='valid')
print(f"Final 100-episode average reward: {running_avg[-1]:.1f}")
plt.plot(running_avg, color='#A7452A')
plt.title('CartPole-v1 Running Average Reward (100-episode window)')
plt.xlabel('Episode'); plt.ylabel('Average Reward')
plt.show()

## Assignment: Train a Mountain Car

Swap `'CartPole-v1'` for `'MountainCar-v0'` (`3` discrete actions: left, coast, right) and tune `discretize()` bin widths and `epsilon` decay.